## Render Map displaying H3 cells of various resolutions

1. Run All
2. Select layers of output map using check boxes

In [ ]:
import json
import folium
import h3
from h3 import LatLngMultiPoly, LatLngPoly
from shapely.geometry import MultiPolygon, Polygon, shape
from shapely.ops import unary_union

resolutions = list(range(1, 7))  # h3 resolutions 1 through 6
m = folium.Map(location=(45, -115), zoom_start=5)


def draw_conus_polygon():
    with open("datasets/conus-states.json", "r") as f:
        state_geo = json.load(f)

    states = [shape(feature["geometry"]) for feature in state_geo["features"]]
    return unary_union(states)


def convert_polygon_to_h3(geom: Polygon | MultiPolygon, resolution: int) -> set[str]:
    """
    Convert a Shapely Polygon or MultiPolygon to H3 cell IDs.
    """
    if isinstance(geom, Polygon):
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [[(lat, lon) for lon, lat in ring.coords] for ring in geom.interiors]
        h3_shape = LatLngPoly(exterior, *holes)
    elif isinstance(geom, MultiPolygon):
        polys = []
        for poly in geom.geoms:
            exterior = [(lat, lon) for lon, lat in poly.exterior.coords]
            holes = [[(lat, lon) for lon, lat in ring.coords] for ring in poly.interiors]
            polys.append(LatLngPoly(exterior, *holes))
        h3_shape = LatLngMultiPoly(*polys)
    else:
        raise TypeError(f"Expected Polygon or MultiPolygon, got {type(geom).__name__}")

    return set(h3.polygon_to_cells_experimental(h3_shape, res=resolution, contain="overlap"))


def build_line_features(cells: set[str]):
    features = []
    for cell in cells:
        boundary = h3.cell_to_boundary(cell)
        coordinates = [[lon, lat] for lat, lon in boundary]
        if coordinates[0] != coordinates[-1]:
            coordinates.append(coordinates[0])
        features.append(
            {
                "type": "Feature",
                "properties": {"cell_id": cell},
                "geometry": {"type": "LineString", "coordinates": coordinates},
            }
        )
    return features


conus = draw_conus_polygon()

for res in resolutions:
    feature_group = folium.FeatureGroup(name=f"resolution {res}", show=False)
    feature_group.add_to(m)

    cells = convert_polygon_to_h3(conus, res)
    geojson = {
        "type": "FeatureCollection",
        "features": build_line_features(cells),
    }

    folium.GeoJson(
        geojson,
        style_function=lambda feature, res=res: {
            "color": "black",
            "weight": 1,
            "opacity": 0.7,
        },
        tooltip=folium.GeoJsonTooltip(fields=["cell_id"], aliases=["cell"]),
    ).add_to(feature_group)

folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
m.save(f"html/cell_resolutions{resolutions}.html")